# Experimento 02 — Distancia geográfica cliente↔vendedor

Este cuaderno extiende el baseline del experimento 01 con una señal nueva: la distancia física entre el cliente y cada vendedor del pedido, derivada de `olist_geolocation_dataset`. La hipótesis es directa: en un país con la extensión de Brasil, la distancia debería explicar una parte estructural del tiempo de entrega.

El pipeline y la configuración son los mismos del baseline (partición 60/20/20 cronológica, `TimeSeriesSplit` con tres pliegues, bootstrap para IC95%, cinco familias de modelos más la línea base trivial). Lo único que cambia es el conjunto de predictores. Los resultados se guardan en `outputs/experimentos/02_distancia_geografica/`.


## 1. Entorno y datos

Resolvemos la raíz del repo para que los imports de `src` funcionen tanto en local como en Colab, y descargamos los CSV de Olist desde la API pública de Kaggle si todavía no están en `data/raw/`.


In [1]:
from pathlib import Path
import os
import subprocess
import sys


In [2]:
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Cde571/LogisTech-Modelos-Simulacion-II.git'

if IN_COLAB:
    root = Path('/content/LogisTech-Modelos-Simulacion-II')
    if not root.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(root)], check=True)
    else:
        subprocess.run(['git', '-C', str(root), 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt')], check=True)
    os.chdir(root)
else:
    root = Path.cwd()
    while root.name in ('experimentos', 'notebook', 'notebooks'):
        root = root.parent
    os.chdir(root)

print('Repositorio detectado:', root.name)


Repositorio detectado: LogisTech-Modelos-Simulacion-II


In [3]:
import urllib.request
import zipfile


In [4]:
raw = root / 'data' / 'raw'
required = raw / 'olist_orders_dataset.csv'

if not required.exists():
    raw.mkdir(parents=True, exist_ok=True)
    archive = root / 'data' / 'olist.zip'
    urllib.request.urlretrieve(
        'https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce',
        archive,
    )
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(raw)

print('CSV disponibles en data/raw:', len(list(raw.glob('*.csv'))))


CSV disponibles en data/raw: 9


## 2. Configuración y dataset enriquecido

Arrancamos con la misma configuración del baseline y construimos el dataset a nivel de pedido. La novedad es `attach_order_geography`, que agrega la coordenada mediana del cliente y la distancia promedio y máxima a los vendedores del pedido. La agregación es segura: la señal está disponible en el instante de la compra y no introduce fuga.


In [5]:
from src.models import ExperimentConfig
from src.preprocessing import build_order_level_dataset, load_olist_tables, modeling_frame
from src.geography import attach_order_geography


In [6]:
MAX_ROWS = None  # Use None para la corrida completa.
config = ExperimentConfig(max_rows=MAX_ROWS, cv_splits=3, bootstrap_iterations=500)
config


ExperimentConfig(random_state=42, max_rows=None, cv_splits=3, bootstrap_iterations=500, min_frequency=20, low_cardinality_max=30, high_cardinality_min=100, selection_criterion='mean')

In [7]:
tables = load_olist_tables(raw)
{name: df.shape for name, df in tables.items()}


{'orders': (99441, 8),
 'items': (112650, 7),
 'customers': (99441, 5),
 'products': (32951, 9),
 'payments': (103886, 5),
 'sellers': (3095, 4),
 'reviews': (99224, 7),
 'geolocation': (1000163, 5),
 'category_translation': (71, 2)}

In [8]:
orders = build_order_level_dataset(tables)
orders = attach_order_geography(orders, tables)
orders.shape


(99441, 37)

In [9]:
geo_cols = [
    'customer_lat', 'customer_lon',
    'mean_customer_seller_distance_km',
    'max_customer_seller_distance_km',
    'sellers_with_coordinates',
]
orders[geo_cols].describe()


,customer_lat,customer_lon,mean_customer_seller_distance_km,max_customer_seller_distance_km,sellers_with_coordinates
count,99163.000000,99163.000000,98177.000000,98177.000000,98666.000000
mean,-21.193324,-46.175700,601.592292,602.897490,1.008564
std,5.609607,4.058439,595.312790,595.794314,0.141151
min,-33.690972,-72.670987,0.000000,0.000000,0.000000
25%,-23.588569,-48.097766,186.518153,188.274065,1.000000
50%,-22.925419,-46.630866,433.769298,435.389942,1.000000
75%,-20.140685,-43.600962,798.024481,800.722774,1.000000
max,42.184003,-8.577855,8677.871551,8677.871551,5.000000


In [10]:
coverage = orders['mean_customer_seller_distance_km'].notna().mean()
print(f'Cobertura de distancia: {coverage:.2%}')


Cobertura de distancia: 98.73%


## 3. Regresión — días hasta la entrega

Corremos el pipeline completo (`tune_models`, `fit_best_and_test`, importancia por permutación sobre validación y comparación con PCA/UMAP). El ranking final se ordena por el score medio de validación cruzada; esa misma lógica es la que el baseline usó para elegir campeón.


In [11]:
from src.experiments import run_task


In [12]:
X_reg, y_reg, dates_reg = modeling_frame(orders, 'delivery_time_days')
X_reg.shape


(96476, 24)

In [13]:
reg = run_task('regression', X_reg, y_reg, dates_reg, config)


/home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


/home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


/home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


/home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


/home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II/.venv/lib/python3.13/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


In [14]:
reg['ranking'][['model', 'train_MAE', 'validation_MAE', 'test_MAE', 'test_ci95_low', 'test_ci95_high']]


,model,train_MAE,validation_MAE,test_MAE,test_ci95_low,test_ci95_high
0,LinearSVR (SVM),5.192525,5.302742,3.923850,3.868065,3.979744
1,MLP (red neuronal),5.245496,5.327233,4.358905,4.297216,4.419671
2,Random Forest (ensamble),4.605530,5.389473,4.144124,4.087904,4.197696
3,Ridge (parametrico),5.467734,5.538340,5.057653,4.997014,5.118160
4,KNN (no parametrico),0.000836,5.577923,4.614408,4.560888,4.677054
5,Baseline trivial (mediana),6.370526,6.799813,4.962445,4.909492,5.020134


In [15]:
reg['feature_importance'].head(12)


,feature,importance_mean,importance_std
0,mean_customer_seller_distance_km,2.484481,0.062053
1,max_customer_seller_distance_km,0.627240,0.017454
2,customer_state,0.535562,0.036122
3,estimated_delivery_days,0.485764,0.027044
4,product_category_name,0.099541,0.013178
5,payment_type,0.039579,0.003761
6,customer_city,0.025939,0.004810
7,total_freight_value,0.024946,0.006663
8,item_count,0.010935,0.004882
9,sellers_with_coordinates,0.008944,0.001637


In [16]:
reg['reductions']


,reduction,model,original_features,reduced_features,reduction_pct,MAE,RMSE,R2
0,PCA_95pct,LinearSVR (SVM),117,24,79.487179,3.962986,5.558836,0.112316
1,PCA_95pct,MLP (red neuronal),117,24,79.487179,4.256506,5.781190,0.039881
2,UMAP_10d,LinearSVR (SVM),117,10,91.452991,4.443407,5.825388,0.025144
3,UMAP_10d,MLP (red neuronal),117,10,91.452991,5.425078,6.685746,-0.284075


## 4. Clasificación — probabilidad de entrega tardía

Misma lógica que la regresión pero sobre el objetivo binario `late_delivery`. La diferencia esperada aquí es menor: en el baseline el *shift* temporal entre validación y test dominaba el resultado, así que la distancia geográfica por sí sola no debería corregirlo.


In [17]:
X_cls, y_cls, dates_cls = modeling_frame(orders, 'late_delivery')
X_cls.shape


(96476, 24)

In [18]:
cls = run_task('classification', X_cls, y_cls, dates_cls, config)


In [19]:
cls['ranking'][['model', 'train_ROC_AUC', 'validation_ROC_AUC', 'test_ROC_AUC', 'test_ci95_low', 'test_ci95_high']]


,model,train_ROC_AUC,validation_ROC_AUC,test_ROC_AUC,test_ci95_low,test_ci95_high
0,Random Forest (ensamble),0.899756,0.727371,0.532568,0.514956,0.549924
1,Logistica (parametrico),0.691622,0.706497,0.727164,0.713872,0.740696
2,MLP (red neuronal),0.682029,0.704353,0.626029,0.610251,0.641481
3,LinearSVC (SVM),0.692149,0.704308,0.721216,0.708357,0.734771
4,KNN (no parametrico),1.000000,0.653752,0.571053,0.553257,0.587355
5,Baseline trivial (clase mayoritaria),0.500000,0.500000,0.500000,0.500000,0.500000


In [20]:
cls['feature_importance'].head(12)


,feature,importance_mean,importance_std
0,estimated_delivery_days,0.093916,0.006065
1,purchase_month,0.062862,0.003374
2,customer_state,0.044120,0.003617
3,total_freight_value,0.024086,0.004409
4,product_category_name,0.021732,0.000664
5,customer_lat,0.020132,0.000826
6,customer_lon,0.019691,0.000418
7,mean_customer_seller_distance_km,0.017984,0.003839
8,customer_city,0.017186,0.000965
9,mean_product_weight_g,0.013558,0.000981


In [21]:
cls['reductions']


,reduction,model,original_features,reduced_features,reduction_pct,balanced_accuracy,precision,recall,F1,ROC_AUC,avg_precision
0,PCA_95pct,Random Forest (ensamble),117,24,79.487179,0.499239,0.052306,0.124388,0.073645,0.593613,0.063048
1,PCA_95pct,Logistica (parametrico),117,24,79.487179,0.690617,0.093268,0.834476,0.167783,0.723899,0.109080
2,UMAP_10d,Random Forest (ensamble),117,10,91.452991,0.491931,0.048156,0.154750,0.073454,0.435010,0.047927
3,UMAP_10d,Logistica (parametrico),117,10,91.452991,0.438287,0.035998,0.248776,0.062895,0.390079,0.043124


## 5. Comparación con el baseline

Cargamos el `summary.json` del experimento 01 y lo ponemos al lado de los números de esta corrida. La diferencia que importa es sobre test; el movimiento en validación no cuenta porque el *shift* temporal ya está documentado.


In [22]:
import json

baseline = json.loads((root / 'outputs/experimentos/01_baseline_pre_compra/summary.json').read_text())
baseline


{'sample_rows': None,
 'cv_splits': 3,
 'bootstrap_iterations': 500,
 'regression': {'champion': 'LinearSVR (SVM)',
  'test_primary': 3.979647162704609,
  'test_ci95': [3.922299705648949, 4.036764568215699]},
 'classification': {'champion': 'MLP (red neuronal)',
  'test_primary': 0.6747671269952074,
  'test_ci95': [0.6603114890517523, 0.6884271227161061]}}

In [23]:
import pandas as pd

comparison = pd.DataFrame(
    [
        {
            'tarea': 'regresión (MAE días)',
            'baseline': baseline['regression']['test_primary'],
            'con distancia': float(reg['ranking'].iloc[0]['test_MAE']),
        },
        {
            'tarea': 'clasificación (ROC-AUC)',
            'baseline': baseline['classification']['test_primary'],
            'con distancia': float(cls['ranking'].iloc[0]['test_ROC_AUC']),
        },
    ]
)
comparison['delta'] = comparison['con distancia'] - comparison['baseline']
comparison


,tarea,baseline,con distancia,delta
0,regresión (MAE días),3.979647,3.923850,-0.055797
1,clasificación (ROC-AUC),0.674767,0.532568,-0.142199


## 6. Persistencia de resultados

Guardamos las cuatro tablas por tarea y un `summary.json` que incluye la cobertura geográfica para dejar documentado el porcentaje de pedidos con coordenadas válidas. El `README.md` de la carpeta discute los hallazgos en prosa.


In [24]:
output = root / 'outputs' / 'experimentos' / '02_distancia_geografica'
output.mkdir(parents=True, exist_ok=True)

for prefix, result in [('regression', reg), ('classification', cls)]:
    for name in ['cv_results', 'ranking', 'feature_importance', 'reductions']:
        result[name].to_csv(output / f'{prefix}_{name}.csv', index=False)

print('Tablas guardadas en', output.relative_to(root))


Tablas guardadas en outputs/experimentos/02_distancia_geografica


In [25]:
def _headline(result, primary_col):
    top = result['ranking'].iloc[0]
    return {
        'champion': top['model'],
        'test_primary': float(top[primary_col]),
        'test_ci95': [float(top['test_ci95_low']), float(top['test_ci95_high'])],
    }

summary = {
    'sample_rows': MAX_ROWS,
    'cv_splits': config.cv_splits,
    'bootstrap_iterations': config.bootstrap_iterations,
    'geo_coverage': float(coverage),
    'regression': _headline(reg, 'test_MAE'),
    'classification': _headline(cls, 'test_ROC_AUC'),
}
(output / 'summary.json').write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8'
)
summary


{'sample_rows': None,
 'cv_splits': 3,
 'bootstrap_iterations': 500,
 'geo_coverage': 0.9872889452036886,
 'regression': {'champion': 'LinearSVR (SVM)',
  'test_primary': 3.9238500219365964,
  'test_ci95': [3.8680654098084446, 3.9797438886485206]},
 'classification': {'champion': 'Random Forest (ensamble)',
  'test_primary': 0.5325684563965212,
  'test_ci95': [0.5149559281104322, 0.5499237273695717]}}